# Analysis of Misclassifications

In [17]:
from pathlib import Path
import json
import pickle

import pandas as pd

In [18]:
# Experiment config

dataset = "darpa2000"
scenario = "s1_inside"

logic_file = "darpa_neg"
train_mode = "scratch" # "scratch" or "pretrained"
dataset_variant = "down" # "original" or "resampled"
window_size = 10

run_id = "20260327_094747"

## Compute Misclassifications

### Load Original Flows

In [19]:
df = pd.read_csv(
    f"../data/interim/{dataset}/{scenario}/flows_labeled/all_flows_labeled.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)
df['t_rel'] = df['start_time'] - df['start_time'].min()

### Load DPL Dataset

In [20]:
def load_dpl_dataset(logic_file, cache_file_name):
    dpl_dataset_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/cache/")
    cache_file_test = dpl_dataset_dir / cache_file_name

    cache = pickle.load(open(cache_file_test, "rb"))
    cache_df = pd.DataFrame(cache)
    cache_df.head()

    return cache_df

In [21]:
cache_file_name = f"{logic_file}_{dataset_variant}_w{window_size}_test.pkl"
cache_df = load_dpl_dataset(logic_file, cache_file_name)

### Map Misclassifications to Original Flows

In [22]:
errors_dir = f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/errors"
experiment_name = f"{logic_file}_{train_mode}_{dataset_variant}_w{window_size}"

errors_file = (
    f"{errors_dir}/"
    f"{experiment_name}_{run_id}.json"
)
    
with open(errors_file) as f:
    errors = json.load(f)

In [23]:
dpl_to_orig = dict(zip(cache_df['dpl_index'], cache_df['orig_index']))

original_indices = []
mis_y_preds = []
mis_y_trues = []

phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
    "phase5": 5,
}

for error in errors:
    dpl_index = error['index']

    original_indices.append(dpl_to_orig[dpl_index])
    y_pred = error['predicted']
    y_true = error['actual']
    mis_y_preds.append(phase_map[y_pred])
    mis_y_trues.append(phase_map[y_true])

mis_df = df.loc[original_indices].copy()
mis_df['y_pred'] = mis_y_preds
mis_df['y_true'] = mis_y_trues

In [24]:
mis_df

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
24136,f24195,9.524417e+08,9.524417e+08,0.121346,202.77.162.213,54792,172.16.115.20,32773,udp,-,...,1,100,1,52,-,17,2,2790.882799,0,2
24689,f25508,9.524419e+08,9.524420e+08,110.206492,172.16.112.194,3,202.77.162.213,3,icmp,-,...,18,2016,0,0,-,1,2,3033.182087,0,2
29065,f29034,9.524432e+08,9.524432e+08,1.605300,202.77.162.213,46959,172.16.115.20,23,tcp,-,...,13,726,13,839,-,6,3,4314.762450,0,3
30791,f125815,9.524438e+08,9.524505e+08,6746.255378,197.182.91.233,47288,172.16.115.20,23,tcp,-,...,3005,157848,1555,90717,-,6,0,4896.435225,4,0
34593,f34589,9.524448e+08,9.524448e+08,0.000781,172.16.112.50,34470,172.16.115.20,53,udp,dns,...,1,61,1,119,-,17,0,5921.560587,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118181,f118164,9.524482e+08,9.524482e+08,0.001025,172.16.113.148,1658,172.16.115.20,53,udp,dns,...,1,63,1,126,-,17,0,9295.727508,5,0
118183,f118166,9.524482e+08,9.524482e+08,0.001434,172.16.115.5,1661,172.16.115.20,53,udp,dns,...,1,58,1,118,-,17,0,9296.338915,5,0
118195,f118172,9.524482e+08,9.524482e+08,0.000794,172.16.117.132,1667,172.16.115.20,53,udp,dns,...,1,58,1,118,-,17,0,9296.808459,5,0
118199,f118174,9.524482e+08,9.524482e+08,0.000786,172.16.117.132,1671,172.16.115.20,53,udp,dns,...,1,58,1,118,-,17,0,9296.959074,5,0


## Misclassification Analysis

### Helper functions

In [25]:
def false_positives(df, phase):
    return df[(df["y_true"] != phase) & (df["y_pred"] == phase)]

def false_negatives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] != phase)]

def true_positives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] == phase)]

### Per-Phase Misclassifications

#### Phase 2

In [26]:
df_fp_2 = false_positives(mis_df, 2)
df_fp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [27]:
df_fn_2 = false_negatives(mis_df, 2)
df_fn_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
24136,f24195,9.524417e+08,9.524417e+08,0.121346,202.77.162.213,54792,172.16.115.20,32773,udp,-,...,1,100,1,52,-,17,2,2790.882799,0,2
24689,f25508,9.524419e+08,9.524420e+08,110.206492,172.16.112.194,3,202.77.162.213,3,icmp,-,...,18,2016,0,0,-,1,2,3033.182087,0,2


In [28]:
df_fn_2["local_orig"]
df_fn_2["local_resp"]

24136    T
24689    F
Name: local_resp, dtype: object

In [29]:
df_tp_2 = true_positives(mis_df, 2)
df_tp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 3

In [30]:
df_fn_3 = false_negatives(mis_df, 3)
df_fn_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
29065,f29034,9.524432e+08,9.524432e+08,1.6053,202.77.162.213,46959,172.16.115.20,23,tcp,-,...,13,726,13,839,-,6,3,4314.76245,0,3


In [31]:
df_fp_3 = false_positives(mis_df, 3)
df_fp_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 4

In [32]:
df_fn_4 = false_negatives(mis_df, 4)
df_fn_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [33]:
df_fp_4 = false_positives(mis_df, 4)
df_fp_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
30791,f125815,9.524438e+08,9.524505e+08,6746.255378,197.182.91.233,47288,172.16.115.20,23,tcp,-,...,3005,157848,1555,90717,-,6,0,4896.435225,4,0


In [34]:
df_fp_4["dport"].value_counts()

dport
23    1
Name: count, dtype: int64

#### Phase 5

In [35]:
df_fn_5 = false_negatives(mis_df, 5)
df_fn_5

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
39483,f39454,9.524464e+08,9.524464e+08,0.000658,172.16.115.20,1020,202.77.162.213,1022,tcp,-,...,2,80,2,80,-,6,5,7528.056140,0,5
39857,f40684,9.524465e+08,9.524465e+08,0.000000,8.138.161.2,4845,131.84.1.31,12277,tcp,-,...,1,40,0,0,-,6,5,7574.669127,0,5
39860,f40687,9.524465e+08,9.524465e+08,0.000000,61.56.80.155,4940,131.84.1.31,31938,tcp,-,...,1,40,0,0,-,6,5,7574.669261,0,5
39861,f40688,9.524465e+08,9.524465e+08,0.000000,61.56.80.155,17061,131.84.1.31,29313,tcp,-,...,1,40,0,0,-,6,5,7574.669284,0,5
39862,f40689,9.524465e+08,9.524465e+08,0.000000,116.107.159.190,17062,131.84.1.31,16367,tcp,-,...,1,40,0,0,-,6,5,7574.669324,0,5
39867,f40694,9.524465e+08,9.524465e+08,0.000000,116.107.159.190,4847,131.84.1.31,12114,tcp,-,...,1,40,0,0,-,6,5,7574.669764,0,5
39868,f40695,9.524465e+08,9.524465e+08,0.000000,45.4.65.127,4848,131.84.1.31,7088,tcp,-,...,1,40,0,0,-,6,5,7574.669830,0,5
39872,f40699,9.524465e+08,9.524465e+08,0.000000,2.178.110.11,17066,131.84.1.31,11586,tcp,-,...,1,40,0,0,-,6,5,7574.670094,0,5
39873,f40700,9.524465e+08,9.524465e+08,0.000000,2.178.110.11,4851,131.84.1.31,2769,tcp,-,...,1,40,0,0,-,6,5,7574.670169,0,5
39874,f40701,9.524465e+08,9.524465e+08,0.000000,65.143.237.207,17067,131.84.1.31,27827,tcp,-,...,1,40,0,0,-,6,5,7574.670283,0,5
